In [ ]:
# 🎯 Sign Language Recognition Training - Google Colab

**Target: 97%+ Accuracy with GPU Acceleration**

This notebook will train your sign language recognition model using Google Colab's free GPU for much faster training.

## 🚀 Step 1: Setup and Installation

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install mediapipe opencv-python numpy matplotlib tqdm

In [ ]:
# Check GPU availability
import torch
print(f"🚀 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎯 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU available, using CPU (will be slower)")

## 📁 Step 2: Upload Your Data

You need to upload your processed data from your local machine to Colab.

In [ ]:
# Create directories
!mkdir -p data/working models utils

print("📁 Directories created")
print("\n📤 Next steps:")
print("1. Upload your 'data.zip' file (containing data/working folder)")
print("2. Upload your 'utils.zip' file (containing utils folder)")
print("3. Run the next cell to upload files")

In [ ]:
from google.colab import files
import zipfile
import os

print("📤 Upload your data files:")
print("1. First upload your 'data.zip' (containing data/working folder)")
print("2. Then upload your 'utils.zip' (containing utils folder)")

# Upload data files
uploaded = files.upload()

# Extract uploaded files
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"📦 Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✅ {filename} extracted successfully")

print("\n📁 Checking uploaded files:")
if os.path.exists('data/working'):
    folders = os.listdir('data/working')
    print(f"✅ Found {len(folders)} data folders: {folders[:5]}...")
else:
    print("❌ data/working folder not found")

if os.path.exists('utils'):
    files = os.listdir('utils')
    print(f"✅ Found {len(files)} utility files: {files}")
else:
    print("❌ utils folder not found")

## 🎯 Step 3: Training Script

This is the optimized training script for Colab with GPU acceleration.

In [ ]:
#!/usr/bin/env python3
"""
Colab Optimized Sign Language Trainer
=====================================

Optimized for Google Colab with GPU acceleration.
"""

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import json
import random
import glob
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

# Simple label encoder
class SimpleLabelEncoder:
    def __init__(self):
        self.classes_ = []
        self.class_to_idx = {}
        self.idx_to_class = {}
    
    def fit(self, labels):
        unique_labels = sorted(list(set(labels)))
        self.classes_ = unique_labels
        self.class_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
        self.idx_to_class = {idx: label for label, idx in self.class_to_idx.items()}
        return self
    
    def fit_transform(self, labels):
        self.fit(labels)
        return self.transform(labels)
    
    def transform(self, labels):
        return [self.class_to_idx[label] for label in labels]
    
    def inverse_transform(self, indices):
        return [self.idx_to_class[idx] for idx in indices]

# Simple train-test split
def simple_train_test_split(sequences, labels, test_size=0.2, random_state=42):
    random.seed(random_state)
    
    label_groups = {}
    for seq, label in zip(sequences, labels):
        if label not in label_groups:
            label_groups[label] = []
        label_groups[label].append(seq)
    
    train_sequences = []
    train_labels = []
    test_sequences = []
    test_labels = []
    
    for label, seqs in label_groups.items():
        random.shuffle(seqs)
        split_idx = int(len(seqs) * (1 - test_size))
        
        train_sequences.extend(seqs[:split_idx])
        train_labels.extend([label] * split_idx)
        
        test_sequences.extend(seqs[split_idx:])
        test_labels.extend([label] * (len(seqs) - split_idx))
    
    return train_sequences, test_sequences, train_labels, test_labels

class ColabDataset(Dataset):
    def __init__(self, sequences, labels, augment=True):
        self.sequences = sequences
        self.labels = labels
        self.augment = augment
        
    def __len__(self):
        return len(self.sequences)
        
    def __getitem__(self, idx):
        sequence = self.sequences[idx]
        
        if self.augment and random.random() < 0.3:
            sequence = self.advanced_augment(sequence)
            
        return torch.FloatTensor(sequence), self.labels[idx]
    
    def advanced_augment(self, sequence):
        augmented = sequence.copy()
        
        # Add controlled noise
        if random.random() < 0.3:
            noise = np.random.normal(0, 0.005, sequence.shape)
            augmented += noise
        
        # Time warping
        if random.random() < 0.2:
            time_scale = random.uniform(0.9, 1.1)
            new_length = int(sequence.shape[0] * time_scale)
            if new_length != sequence.shape[0]:
                indices = np.linspace(0, sequence.shape[0]-1, new_length)
                interpolated = []
                for i in range(sequence.shape[1]):
                    interpolated.append(np.interp(indices, range(sequence.shape[0]), sequence[:, i]))
                augmented = np.column_stack(interpolated)
                
                if augmented.shape[0] < sequence.shape[0]:
                    padding = np.tile(augmented[-1:], (sequence.shape[0] - augmented.shape[0], 1))
                    augmented = np.vstack([augmented, padding])
                else:
                    augmented = augmented[:sequence.shape[0]]
        
        # Spatial augmentation
        if random.random() < 0.2:
            scale = random.uniform(0.95, 1.05)
            augmented *= scale
        
        return augmented

class ColabTrainer:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"🚀 Using device: {self.device}")
        
        os.makedirs("models", exist_ok=True)
        
        # Optimized config for Colab
        self.config = {
            "model_architecture": {
                "hidden_dim": 512,
                "num_heads": 16,
                "num_layers": 8,
                "dropout": 0.1,
                "sequence_length": 30
            },
            "training": {
                "batch_size": 32,        # CORRECTED: Was 64 (too large)
                "learning_rate": 0.0001, # CORRECTED: Was 0.001 (too high)
                "epochs": 300,
                "patience": 25,          # CORRECTED: Was 50 (too high)
                "weight_decay": 0.01,
                "gradient_clip": 1.0
            }
        }
    
    def load_data(self):
        print("📊 Loading ASL dataset...")
        
        sequences = []
        labels = []
        
        train_folders = glob.glob("data/working/*_train")
        print(f"Found {len(train_folders)} training folders")
        if len(train_folders) == 0:
            print("❌ No training folders found!")
            print("   Current working directory:", os.getcwd())
            print("   Checking if data/working exists:", os.path.exists("data/working"))
            if os.path.exists("data/working"):
                print("   Contents of data/working:", os.listdir("data/working"))
            return [], []
        print(f"   Folders: {[os.path.basename(f) for f in train_folders[:5]]}...")
        
        for folder in tqdm(train_folders, desc="Loading data"):
            folder_name = os.path.basename(folder)
            letter = folder_name.split("_")[0]
            sequence_files = glob.glob(os.path.join(folder, "*.npy"))
            
            for seq_file in sequence_files[:2000]:  # Use more data per letter
                try:
                    sequence = np.load(seq_file)
                    if sequence.shape == (30, 153):
                        sequences.append(sequence)
                        labels.append(letter)
                except Exception as e:
                    continue
        
        print(f"✅ Loaded {len(sequences)} sequences")
        print(f"   Labels: {sorted(set(labels))}")
        
        return sequences, labels
    
    def prepare_data(self, sequences, labels):
        print("🔧 Preparing data...")
        
        label_encoder = SimpleLabelEncoder()
        encoded_labels = label_encoder.fit_transform(labels)
        
        label_mapping = {str(i): label for i, label in enumerate(label_encoder.classes_)}
        with open('models/label_mapping.json', 'w') as f:
            json.dump(label_mapping, f, indent=2)
        
        train_sequences, test_sequences, train_labels, test_labels = simple_train_test_split(
            sequences, encoded_labels, test_size=0.2, random_state=42
        )
        
        train_dataset = ColabDataset(train_sequences, train_labels, augment=True)
        test_dataset = ColabDataset(test_sequences, test_labels, augment=False)
        
        train_loader = DataLoader(train_dataset, batch_size=self.config["training"]["batch_size"], shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=self.config["training"]["batch_size"], shuffle=False)
        
        print(f"   Training samples: {len(train_sequences)}")
        print(f"   Test samples: {len(test_sequences)}")
        
        return train_loader, test_loader
    
    def create_model(self, num_classes):
        from utils.translator import GestureTransformer
        
        arch = self.config["model_architecture"]
        
        model = GestureTransformer(
            input_dim=153,
            hidden_dim=arch["hidden_dim"],
            num_heads=arch["num_heads"],
            num_layers=arch["num_layers"],
            num_classes=num_classes
        ).to(self.device)
        
        print(f"🏗️  Model created:")
        print(f"   Hidden dim: {arch['hidden_dim']}")
        print(f"   Attention heads: {arch['num_heads']}")
        print(f"   Layers: {arch['num_layers']}")
        print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
        
        return model
    
    def train_epoch(self, model, train_loader, optimizer, criterion):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for batch_idx, (data, target) in enumerate(tqdm(train_loader, desc="Training")):
            data, target = data.to(self.device), target.to(self.device)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), self.config["training"]["gradient_clip"])
            
            optimizer.step()
            
            total_loss += loss.item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            total += target.size(0)
        
        return total_loss / len(train_loader), 100. * correct / total
    
    def validate_epoch(self, model, test_loader, criterion):
        model.eval()
        total_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for data, target in tqdm(test_loader, desc="Validation"):
                data, target = data.to(self.device), target.to(self.device)
                output = model(data)
                loss = criterion(output, target)
                
                total_loss += loss.item()
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()
                total += target.size(0)
        
        return total_loss / len(test_loader), 100. * correct / total
    
    def plot_training_history(self, train_losses, val_losses, train_accs, val_accs):
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 2, 1)
        plt.plot(train_losses, label='Train Loss')
        plt.plot(val_losses, label='Val Loss')
        plt.title('Training and Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        
        plt.subplot(1, 2, 2)
        plt.plot(train_accs, label='Train Acc')
        plt.plot(val_accs, label='Val Acc')
        plt.title('Training and Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy (%)")
        plt.legend()
        
        plt.tight_layout()
        plt.savefig('models/training_history.png')
        plt.show()
    
    def train(self):
        print("🎯 Colab Training (Target: 97%+)")
        print("=" * 50)
        
        sequences, labels = self.load_data()
        train_loader, test_loader = self.prepare_data(sequences, labels)
        model = self.create_model(len(set(labels)))
        
        train_config = self.config["training"]
        
        optimizer = optim.AdamW(
            model.parameters(),
            lr=train_config["learning_rate"],
            weight_decay=train_config["weight_decay"]
        )
        
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=10, factor=0.5, verbose=True
        )
        
        criterion = nn.CrossEntropyLoss()
        
        best_accuracy = 0
        patience_counter = 0
        train_losses = []
        val_losses = []
        train_accs = []
        val_accs = []
        
        print("🎯 Starting training...")
        print("=" * 50)
        
        for epoch in range(train_config["epochs"]):
            train_loss, train_acc = self.train_epoch(model, train_loader, optimizer, criterion)
            val_loss, val_acc = self.validate_epoch(model, test_loader, criterion)
            
            scheduler.step(val_loss)
            
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            train_accs.append(train_acc)
            val_accs.append(val_acc)
            
            if val_acc > best_accuracy:
                best_accuracy = val_acc
                patience_counter = 0
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'epoch': epoch,
                    'best_accuracy': best_accuracy,
                    'config': self.config
                }, 'models/best_gesture_model.pt')
                print(f"💾 New best model: {best_accuracy:.2f}%")
            else:
                patience_counter += 1
            
            if patience_counter >= train_config["patience"]:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{train_config['epochs']}]")
                print(f"   Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
                print(f"   Val: Loss={val_loss:.4f}, Acc={val_acc:.2f}%")
                print(f"   LR: {optimizer.param_groups[0]['lr']:.6f}")
                print("-" * 40)
        
        self.plot_training_history(train_losses, val_losses, train_accs, val_accs)
        
        print(f"🎉 Training completed! Best accuracy: {best_accuracy:.2f}%")
        
        if best_accuracy >= 97.0:
            print("🎯 Target achieved! 97%+ accuracy reached!")
        else:
            print("⚠️  Target not reached. Consider:")
            print("   - More training data")
            print("   - Longer training time")
            print("   - Model adjustments")

In [ ]:
# Create trainer instance
trainer = ColabTrainer()
print("✅ Trainer created successfully")
print(f"🚀 Device: {trainer.device}")
print(f"📊 Config: {trainer.config['model_architecture']['hidden_dim']} hidden dim, {trainer.config['model_architecture']['num_layers']} layers")

## 🚀 Step 4: Start Training

Run this cell to start the training process. This will take 2-4 hours but will be much faster than local training.

In [ ]:
# Start training
trainer.train()

## 📥 Step 5: Download Results

After training completes, download your trained model and results.

In [ ]:
# Download trained model and results
from google.colab import files
import zipfile

# Create zip file with results
with zipfile.ZipFile('trained_model.zip', 'w') as zipf:
    if os.path.exists('models/best_gesture_model.pt'):
        zipf.write('models/best_gesture_model.pt')
    if os.path.exists('models/label_mapping.json'):
        zipf.write('models/label_mapping.json')
    if os.path.exists('models/training_history.png'):
        zipf.write('models/training_history.png')

print("📦 Model files zipped successfully")
print("📥 Downloading trained model...")
files.download('trained_model.zip')

print("\n🎉 Training complete!")
print("📋 Next steps:")
print("1. Download the 'trained_model.zip' file")
print("2. Extract it to your local 'models' folder")
print("3. Test with: python test_model.py")
print("4. Start web interface: python app.py")